# ⚡ Notebook 08: Training on Apple Silicon

Memory monitoring, mixed precision, gradient accumulation, JIT compilation, and memory budgeting for Apple Silicon ML.

**Prerequisites:** Notebooks 01-07

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Memory Monitoring

MLX provides `mx.metal.get_active_memory()`, `mx.metal.get_peak_memory()`, and `mx.metal.get_cache_memory()` for tracking GPU memory usage.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import time
import math
import matplotlib.pyplot as plt
import numpy as np

class MemoryMonitor:
    def __init__(self):
        self.history = []
    def snapshot(self, label=''):
        active = mx.metal.get_active_memory() / 1e6
        peak = mx.metal.get_peak_memory() / 1e6
        cache = mx.metal.get_cache_memory() / 1e6
        self.history.append({'label': label, 'active_mb': active, 'peak_mb': peak, 'cache_mb': cache})
        return {'active_mb': active, 'peak_mb': peak, 'cache_mb': cache}
    def plot(self):
        labels = [h['label'] for h in self.history]
        active = [h['active_mb'] for h in self.history]
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(active, 'b-o', label='Active Memory (MB)')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
        ax.set_ylabel('Memory (MB)')
        ax.set_title('Metal GPU Memory Usage')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        return fig

mon = MemoryMonitor()
snap = mon.snapshot('baseline')
print(f'Baseline memory: {snap["active_mb"]:.1f} MB active, {snap["peak_mb"]:.1f} MB peak')

## Mixed Precision Training (Deep Dive)

Compare float32 vs bfloat16 for training speed, memory usage, and loss stability.

⚡ **bfloat16** has the same exponent range as float32 (8 bits) but fewer mantissa bits (7 vs 23) — more stable for training than float16 while using half the memory.

🎯 **Interview tip:** Most modern LLM training (GPT-4, LLaMA, Gemma) uses bfloat16. Google TPUs and Apple Silicon both have native bfloat16 support.

| Format | Exponent | Mantissa | Range | Precision |
|--------|----------|----------|-------|-----------|
| float32 | 8 bits | 23 bits | ±3.4e38 | ~7 decimal digits |
| float16 | 5 bits | 10 bits | ±6.5e4 | ~3 decimal digits |
| bfloat16 | 8 bits | 7 bits | ±3.4e38 | ~2 decimal digits |

⚠️ **Pitfall:** float16 can overflow during training (range only ±65504). bfloat16 avoids this.

In [ ]:
from utils.apple_silicon_training import MixedPrecisionTrainer

# Build a small model for comparison
class SmallTransformerBlock(nn.Module):
    def __init__(self, d=64, vocab=256):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.norm = nn.RMSNorm(d)
        self.head = nn.Linear(d, vocab)
    def __call__(self, x):
        h = self.embed(x)
        h = h + self.fc2(nn.silu(self.fc1(self.norm(h))))
        return self.head(h)

def ce_loss(model, batch):
    x, y = batch
    logits = model(x)
    return nn.losses.cross_entropy(logits, y, reduction="mean")

# Create training data
x_train = mx.random.randint(0, 256, shape=(8, 32))
y_train = mx.random.randint(0, 256, shape=(8, 32))
mx.eval(x_train, y_train)

results = MixedPrecisionTrainer.compare_dtypes(
    model_factory=SmallTransformerBlock,
    loss_fn=ce_loss,
    batch=(x_train, y_train),
    n_steps=5,
)

print("⚡ Mixed Precision Comparison:")
print(f"{'Metric':<20} {'float32':>12} {'bfloat16':>12}")
print("-" * 46)
print(f"{'Avg step time (ms)':<20} {results['float32']['avg_time']*1000:>12.2f} {results['bfloat16']['avg_time']*1000:>12.2f}")
print(f"{'Final loss':<20} {results['float32']['losses'][-1]:>12.4f} {results['bfloat16']['losses'][-1]:>12.4f}")
print(f"{'Memory delta (MB)':<20} {results['float32']['memory_mb']:>12.1f} {results['bfloat16']['memory_mb']:>12.1f}")
speedup = results['float32']['avg_time'] / max(results['bfloat16']['avg_time'], 1e-9)
print(f"\n💡 bfloat16 speedup: {speedup:.2f}x")

## Gradient Accumulation (Deep Dive)

When batch size is limited by memory, accumulate gradients over K micro-batches to simulate a larger effective batch size.

🧠 **Intuition:** Gradient accumulation is like taking multiple small sips instead of one big gulp — you process data in small batches but accumulate the learning signal before updating, achieving the same effect as a large batch.

💡 **effective_batch_size = micro_batch_size × accumulation_steps**

🎯 **Interview tip:** Gradient accumulation is essential when GPU memory limits batch size but you need large effective batches for stable training. The key insight is that **averaging gradients over K micro-batches is mathematically equivalent to computing the gradient on the full batch** — because the gradient of a mean is the mean of the gradients.

⚠️ **Critical:** You must *average* (not sum) the accumulated gradients to maintain equivalence with the large-batch gradient.

In [ ]:
from utils.apple_silicon_training import GradientAccumulator
from mlx.utils import tree_flatten
import numpy as np

# --- Build a tiny model for demonstration ---
class TinyFFN(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.fc1 = nn.Linear(d, 64)
        self.fc2 = nn.Linear(64, d)
    def __call__(self, x):
        return self.fc2(nn.relu(self.fc1(x)))

def mse_loss(model, batch):
    x, y = batch
    return mx.mean((model(x) - y) ** 2)

# Create data: full batch of 16, split into 4 micro-batches of 4
mx.random.seed(42)
x_full = mx.random.normal(shape=(16, 32))
y_full = mx.random.normal(shape=(16, 32))
mx.eval(x_full, y_full)

# --- Full-batch gradient (ground truth) ---
model_full = TinyFFN()
mx.eval(model_full.parameters())

# Copy weights for fair comparison
model_accum = TinyFFN()
model_accum.load_weights(list(tree_flatten(model_full.parameters())))
mx.eval(model_accum.parameters())

loss_grad_fn = nn.value_and_grad(model_full, mse_loss)
full_loss, full_grads = loss_grad_fn(model_full, (x_full, y_full))
mx.eval(full_loss, full_grads)

# --- Accumulated micro-batch gradients ---
micro_batches = [(x_full[i*4:(i+1)*4], y_full[i*4:(i+1)*4]) for i in range(4)]
accum_loss, accum_grads = GradientAccumulator.accumulate(
    model_accum, mse_loss, micro_batches, accum_steps=4
)

# --- Compare ---
print(f"Full-batch loss:  {full_loss.item():.6f}")
print(f"Accumulated loss: {accum_loss:.6f}")
print(f"\n💡 Gradient comparison (should be near-zero differences):")
for (k, g_f), (_, g_a) in zip(tree_flatten(full_grads), tree_flatten(accum_grads)):
    mx.eval(g_f, g_a)
    max_diff = np.max(np.abs(np.array(g_f) - np.array(g_a)))
    print(f"  {k:30s}  max|diff| = {max_diff:.2e}")
print(f"\n⚡ Memory: same as batch_size=4, but effective batch_size=16!")

## mx.compile() JIT Compilation (Deep Dive)

`mx.compile()` fuses multiple operations into a single optimized kernel, reducing Metal kernel launch overhead and memory traffic.

🧠 **What is kernel fusion?** Without compilation, each operation (matmul, softmax, add) launches a separate GPU kernel — each with its own overhead of reading inputs from memory and writing outputs back. Kernel fusion combines multiple operations into one kernel, so intermediate results stay in fast on-chip memory instead of making round-trips to main memory. Think of it like combining multiple errands into one trip instead of driving home between each stop.

⚡ **Key benefit:** Repeated operations (like training steps) get compiled once and run faster on subsequent calls.

🎯 **Interview tip:** JIT compilation is similar to `torch.compile()` in PyTorch or `jax.jit()` in JAX. MLX's lazy evaluation makes compilation particularly effective — the entire computation graph is visible before execution.

In [ ]:
from utils.apple_silicon_training import compile_benchmark

# Benchmark a matmul chain with and without mx.compile()
def matmul_chain(x):
    for _ in range(5):
        x = x @ x.T
        x = mx.softmax(x, axis=-1)
    return x

x_bench = mx.random.normal((256, 256))
mx.eval(x_bench)

result = compile_benchmark(matmul_chain, x_bench, n_iters=20, warmup=5)

print(f"⚡ mx.compile() Benchmark:")
print(f"  Uncompiled: {result['uncompiled_ms']:.2f} ms/iter")
print(f"  Compiled:   {result['compiled_ms']:.2f} ms/iter")
print(f"  Speedup:    {result['speedup']:.2f}x")
print(f"\n💡 Compiled functions fuse operations → fewer kernel launches → faster execution")

## Memory Budget Calculator (Deep Dive)

Estimate memory for weights, gradients, optimizer states, activations, and KV-cache.

🎯 **Interview tip:** For Adam optimizer, you need **3× the model weights** in memory:
- 1× for parameters (in training dtype)
- 1× for gradients (same dtype)
- 2× for Adam states (m and v, always float32)

⚠️ **Activations** scale with `batch_size × seq_len` — this is usually the OOM culprit, not the model weights!

In [ ]:
from utils.apple_silicon_training import MemoryBudgetCalculator

# Compute budget for a small GPT-like model
budget = MemoryBudgetCalculator.compute_budget(
    d_model=768, n_layers=12, n_heads=12, d_ff=3072,
    batch_size=16, seq_len=512, vocab_size=32000, dtype_bytes=2,
)

print("🎯 Memory Budget (GPT-small, bf16, batch=16, seq=512):")
print(f"  Parameters:    {budget['params_mb']:>8.1f} MB")
print(f"  Gradients:     {budget['grads_mb']:>8.1f} MB")
print(f"  Optimizer:     {budget['optimizer_mb']:>8.1f} MB  (Adam m+v in fp32)")
print(f"  Activations:   {budget['activations_mb']:>8.1f} MB")
print(f"  ─────────────────────────────")
print(f"  Total (train): {budget['total_mb']:>8.1f} MB  ({budget['total_mb']/1024:.2f} GB)")
print(f"  KV-cache:      {budget['kv_cache_mb']:>8.1f} MB  (inference only)")
print(f"  Total params:  {budget['total_params']:>10,}")

# Plot stacked bar chart
fig = MemoryBudgetCalculator.plot_stacked_bar(budget, title="GPT-small Memory Budget")
plt.show()

## Memory Profiler (Deep Dive)

Track memory usage at each phase of a training step: baseline → forward+backward → gradient materialization → optimizer update → cleanup.

💡 This reveals where memory peaks occur and helps identify optimization opportunities.

In [ ]:
from utils.apple_silicon_training import MemoryProfiler

# Profile a training step on our small model
profile_model = SmallTransformerBlock()
mx.eval(profile_model.parameters())
profile_opt = optim.Adam(learning_rate=1e-3)

timeline = MemoryProfiler.profile_training_step(
    model=profile_model,
    loss_fn=ce_loss,
    batch=(x_train, y_train),
    optimizer=profile_opt,
)

print("⚡ Memory Profile (per training phase):")
for snap in timeline.snapshots:
    print(f"  {snap.phase:25s}  active={snap.active_mb:>8.1f} MB  peak={snap.peak_mb:>8.1f} MB")

# Plot timeline
fig = MemoryProfiler.plot_memory_timeline(timeline)
plt.show()

## OOM Recovery: Auto Batch Size Reduction

⚠️ Out-of-memory (OOM) errors are the most common failure mode when training on Apple Silicon. Instead of crashing, we can **automatically halve the batch size** until the training step fits in memory.

💡 This is a production pattern used in frameworks like HuggingFace Accelerate and DeepSpeed.

In [ ]:
from utils.apple_silicon_training import auto_reduce_batch_size

# Demonstrate OOM recovery with a function that "fails" at large batch sizes
def simulated_train_step(batch_size):
    """Simulate a training step that OOMs above batch_size=64."""
    if batch_size > 64:
        raise RuntimeError("Metal allocation failed: out of memory")
    x = mx.random.normal(shape=(batch_size, 256))
    y = x @ mx.random.normal(shape=(256, 256))
    mx.eval(y)
    print(f"  ✅ batch_size={batch_size} succeeded")

print("⚠️ OOM Recovery Demo (simulated OOM above batch_size=64):")
working_bs = auto_reduce_batch_size(simulated_train_step, initial_batch_size=256)
print(f"\n💡 Found working batch_size = {working_bs}")
print(f"   Use gradient accumulation with {256 // working_bs} steps to recover effective batch size")

## 📜 History & Alternatives: Optimizers — SGD → Adam → AdamW → Lion → Schedule-Free

The evolution of optimizers mirrors the evolution of deep learning itself — each generation solved a critical limitation of the previous one.

### Timeline

| Year | Optimizer | Key Innovation | Who |
|------|-----------|---------------|-----|
| 1951 | **SGD** | Stochastic gradient descent — sample-based updates | Robbins & Monro |
| 1986 | **SGD + Momentum** | Exponential moving average of gradients for acceleration | Rumelhart, Hinton, Williams |
| 2011 | **Adagrad** | Per-parameter adaptive learning rates | Duchi, Hazan, Singer |
| 2012 | **RMSProp** | Fix Adagrad's decaying learning rate with moving average | Hinton (unpublished lecture) |
| 2014 | **Adam** | Combine momentum + adaptive rates with bias correction | Kingma & Ba |
| 2017 | **AdamW** | Decouple weight decay from gradient update — fixes L2 regularization in Adam | Loshchilov & Hutter |
| 2023 | **Lion** | Sign-based optimizer — uses only sign(momentum), 50% less memory than Adam | Chen et al. (Google Brain) |
| 2024 | **Schedule-Free** | Eliminate LR schedules entirely — automatic annealing via averaging | Defazio et al. (Meta) |

### 💡 Key Insights

**Why Adam dominates:** Adam combines the best of momentum (SGD+M) and adaptive learning rates (RMSProp). The bias correction terms handle the cold-start problem elegantly.

**Why AdamW matters:** The original Adam paper applied weight decay *inside* the gradient update, which interacts poorly with adaptive learning rates. AdamW applies weight decay *directly to weights*, which is mathematically equivalent to proper L2 regularization. Nearly all modern LLM training uses AdamW.

**Lion's insight:** You don't need the full magnitude of the momentum — just its *sign*. This reduces optimizer state from 2 tensors (m, v) to 1 tensor (m), cutting optimizer memory by ~50%.

**Schedule-Free's promise:** Instead of hand-tuning warmup steps and cosine decay, maintain a running average of iterates that automatically provides the benefits of scheduling. Still experimental but promising for reducing hyperparameter tuning.

### 🎯 Interview Tips

- "What optimizer would you use for LLM training?" → **AdamW** with cosine LR schedule and linear warmup. This is the standard for GPT-3, LLaMA, Gemma, etc.
- "Why not plain Adam?" → Weight decay in Adam is broken — it scales with the adaptive learning rate. AdamW fixes this.
- "What about SGD?" → SGD with momentum can match Adam's final performance but requires much more careful LR tuning and longer training. Not practical for LLMs.
- "What's new in 2024-2025?" → Lion (memory-efficient) and Schedule-Free (no LR schedule needed) are the frontiers.

## Summary

This notebook covered production-grade Apple Silicon training techniques:

1. **Memory Monitoring** — `mx.metal.get_active_memory()` for real-time GPU tracking
2. **Mixed Precision** — bfloat16 for 2× memory savings with float32-range stability
3. **Gradient Accumulation** — simulate large batches with micro-batch accumulation (mathematically equivalent)
4. **mx.compile()** — JIT compilation for kernel fusion and reduced launch overhead
5. **Memory Budget Calculator** — predict memory needs before training (params + grads + optimizer + activations)
6. **Memory Profiler** — per-phase memory tracking to find optimization opportunities
7. **OOM Recovery** — automatic batch size reduction when memory is exhausted

⚡ All implementations use MLX exclusively on Apple Silicon unified memory.

**Next:** Notebook 09 — Modern Architectures (LLaMA, Mistral, Gemma)